In [1]:
from google.colab import files

uploaded = files.upload()

Saving messy_student_data_203.csv to messy_student_data_203.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("messy_student_data_203.csv")

print(df.head())
print(df.shape)

  student_id study_hours_per_day attendance_percentage previous_score  \
0       S190                 0.8                  87.7           89.0   
1       S096                 2.9                  72.9           82.0   
2       S026                 1.9                  92.9           82.0   
3       S039                 NaN                  63.3           87.0   
4       S150                 4.0                   NaN           78.0   

  sleep_hours_per_day  
0                 8.8  
1                 5.2  
2                 4.6  
3                 7.3  
4           8.5 hours  
(203, 5)


In [3]:
# Remove duplicate rows
df = df.drop_duplicates()

# Select numerical columns
cols = [
    "study_hours_per_day",
    "attendance_percentage",
    "previous_score",
    "sleep_hours_per_day"
]

# Convert values to strings
df[cols] = df[cols].astype("string")

# Remove unwanted words and symbols
for col in cols:
    df[col] = df[col].str.replace(
        r"hrs|hours|marks|%|about",
        "",
        regex=True
    ).str.strip()

# Replace written numbers
word_numbers = {
    "eighty": "80",
    "seventy": "70",
    "six": "6"
}

df[cols] = df[cols].replace(word_numbers)

# Convert to numeric
for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove impossible values
df.loc[~df["study_hours_per_day"].between(0, 16),
       "study_hours_per_day"] = np.nan

df.loc[~df["attendance_percentage"].between(0, 100),
       "attendance_percentage"] = np.nan

df.loc[~df["previous_score"].between(0, 100),
       "previous_score"] = np.nan

df.loc[~df["sleep_hours_per_day"].between(0, 16),
       "sleep_hours_per_day"] = np.nan

# Fill missing values with median
df[cols] = df[cols].fillna(df[cols].median())

print(df[cols].isnull().sum())

study_hours_per_day      0
attendance_percentage    0
previous_score           0
sleep_hours_per_day      0
dtype: int64


In [4]:
np.random.seed(42)

study_score = df["study_hours_per_day"] / 8 * 100
attendance = df["attendance_percentage"]
previous = df["previous_score"]
sleep_score = df["sleep_hours_per_day"] / 8 * 100

df["final_exam_score"] = (
    0.30 * study_score +
    0.25 * attendance +
    0.35 * previous +
    0.10 * sleep_score +
    np.random.normal(0, 4, len(df))
).clip(0, 100)

df[["study_hours_per_day",
    "attendance_percentage",
    "previous_score",
    "sleep_hours_per_day",
    "final_exam_score"]].head()

,study_hours_per_day,attendance_percentage,previous_score,sleep_hours_per_day,final_exam_score
0,0.8,87.7,89.0,8.8,69.061857
1,2.9,72.9,82.0,5.2,63.746943
2,1.9,92.9,82.0,4.6,67.390754
3,4.3,63.3,87.0,7.3,77.617119
4,4.0,71.8,78.0,8.5,69.938387


In [5]:
X = df[cols]

y = df["final_exam_score"]

print(X.head())
print(y.head())

   study_hours_per_day  attendance_percentage  previous_score  \
0                  0.8                   87.7            89.0   
1                  2.9                   72.9            82.0   
2                  1.9                   92.9            82.0   
3                  4.3                   63.3            87.0   
4                  4.0                   71.8            78.0   

   sleep_hours_per_day  
0                  8.8  
1                  5.2  
2                  4.6  
3                  7.3  
4                  8.5  
0    69.061857
1    63.746943
2    67.390754
3    77.617119
4    69.938387
Name: final_exam_score, dtype: Float64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 158
Testing records: 40


In [7]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [8]:
from sklearn.metrics import mean_absolute_error, r2_score

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Absolute Error:", mae)
print("R2 Score:", r2)

Mean Absolute Error: 3.8505705439399747
R2 Score: 0.8319784875404769


In [9]:
new_student = pd.DataFrame({
    "study_hours_per_day": [6],
    "attendance_percentage": [90],
    "previous_score": [75],
    "sleep_hours_per_day": [8]
})

predicted_score = model.predict(new_student)

print("Predicted Exam Score:", round(predicted_score[0], 2))

Predicted Exam Score: 80.06
